In [38]:
import os
os.chdir("/Users/naginilandarhi/DSAICourse/Prof_Shankar_Venkatagiri/LinAlg4DataScience/faker_data_synthesizer")

In [39]:
import pandas as pd
import numpy as np

In [40]:
print(os.getcwd())

/Users/naginilandarhi/DSAICourse/Prof_Shankar_Venkatagiri/LinAlg4DataScience/faker_data_synthesizer


In [41]:
r10000 = pd.read_csv('rows10000/synthetic_data_10000_rows_10_cols_3_clusters.csv')

In [42]:
r10000.head()

,ID,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9
0,1ff49b7889464e85b59cde66bacfb3d0,605.167893,-72.794878,199.200935,144.401980,707.382550,641.847832,878.814839,-5.731675,365.403889
1,3b982ef8daf64a26946d3f31fc377a4c,-65.461067,146.140430,449.613954,-86.467961,123.440441,619.089755,506.331920,136.031782,536.423803
2,a28defe39bf0427392476f57a5e5a5ab,795.317503,-83.703844,789.654568,673.330594,283.452007,70.422521,943.779724,266.830867,8.150915
3,50c187fcce174b4e8837b8a3d261a7ab,600.669154,-73.051151,202.935018,147.600811,708.861798,644.110464,880.130394,-8.254955,364.132780
4,23bed01d43cf4fdea4933b83757750a9,790.548508,-84.593729,785.719199,676.453693,279.478581,67.137398,957.206183,292.139707,-0.783434


In [43]:
r10000.dtypes

ID        object
col_1    float64
col_2    float64
col_3    float64
col_4    float64
col_5    float64
col_6    float64
col_7    float64
col_8    float64
col_9    float64
dtype: object

In [44]:
data_matrix_a = r10000.drop(columns=['ID']).to_numpy(dtype=np.float32)

In [45]:
data_matrix_a.shape

(10000, 9)

In [46]:
ids = r10000["ID"].astype(str).to_numpy()

I need to do is to deal with id column. I will drop the id column from the data matrix. I will keep only the feature columns in the data matrix.

Now I need to identify which cluster each row belongs to. I do not want to use KMeans from sklearn. I want to implement KMeans from scratch. I will use the following steps:
1. Randomly select k rows from the data matrix as the initial centroids.
2. Assign each row to the nearest centroid based on Euclidean distance.
3. Recalculate the centroids as the mean of the rows assigned to each centroid.
4. Repeat steps 2 and 3 until convergence (i.e., the centroids do not change significantly between iterations). 

In [47]:
import time

In [48]:
def kmeans_cpu(X,
               K,
               initial_centroids=None,
               max_iters=100,
               tol=1e-4,
               seed=42):
    N, D = X.shape
    if initial_centroids is None:
        rng = np.random.default_rng(seed)
        indices = rng.choice(N, K, replace=False)
        centroids = X[indices].copy()

    else:
        centroids = initial_centroids.copy()
        centroids = centroids.astype(np.float32)

    assignments = np.full(N, -1, dtype=np.int32)   
    start_time = time.perf_counter()

    for iteration in range(max_iters):
        distances = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        new_assignments = np.argmin(distances, axis=1)

        if np.array_equal(assignments, new_assignments):
            print(f"Converged after {iteration} iterations.")
            break

        assignments = new_assignments
        new_centroids = np.empty_like(centroids)

        for k in range(K):
            assigned_points = X[assignments == k]
            if len(assigned_points) > 0:
                new_centroids[k] = assigned_points.mean(axis=0)
                new_centroids[k] = new_centroids[k].astype(np.float32)

        if np.array_equal(centroids, new_centroids):
            print(f"Converged after {iteration} iterations.")
            break

        movement = np.linalg.norm(
            new_centroids - centroids,
            axis=1
        )

        max_movement = np.max(movement)
        centroids = new_centroids
        if max_movement < tol:
            break

    end_time = time.perf_counter()
    execution_time = end_time - start_time
    return {
        "assignments": assignments,
            "centroids": centroids,
            "execution_time": execution_time,
            "iterations": iteration + 1,
            "max_movement": max_movement
        }        


In [49]:
rng = np.random.default_rng(42)
initial_centroids = rng.choice(len(data_matrix_a), size=3, replace=False)
initial_centroids = data_matrix_a[initial_centroids]
print("Initial centroids:\n", initial_centroids)

Initial centroids:
 [[602.00745  -74.00422  204.63234  147.27855  710.04004  645.25775
  883.3031    -9.001872 365.93143 ]
 [-63.968403 149.66426  450.36392  -70.27071  114.135635 614.9672
  496.9724   139.69873  534.4352  ]
 [795.5782   -88.10335  787.96954  675.1455   275.20212   66.08927
  956.2868   264.6806     9.823542]]


In [50]:
result = kmeans_cpu(data_matrix_a, K=3, initial_centroids=initial_centroids, max_iters=100, tol=1e-4, seed=42)
print(f"Execution time: {result['execution_time']:.4f} seconds")
print(f"Iterations: {result['iterations']}")
print(f"Max movement: {result['max_movement']:.6f}")
print(f"Centroids:\n{result['centroids']}")

Converged after 1 iterations.
Execution time: 0.0021 seconds
Iterations: 2
Max movement: 18.795660
Centroids:
[[603.39404   -72.5247    202.52405   145.54764   710.0657    644.3766
  881.3795     -4.339006  364.0983   ]
 [-67.268555  140.41817   455.77057   -71.02961   118.621475  614.9424
  499.5241    142.45778   548.3219   ]
 [790.58856   -92.856865  786.4874    668.06396   274.30948    70.903885
  953.0861    270.31824     1.9503968]]


In [51]:
output = pd.DataFrame({
    "ID": ids,
    "Cluster": result["assignments"]
})

output.to_csv("kmeans_output.csv", index=False)

In [52]:
np.savetxt(
    "initial_centroids.csv",
    initial_centroids,
    delimiter=","
)

In [53]:
def representative_points(
    X,
    ids,
    centroids,
    assignments
):
    representatives = []

    for k in range(len(centroids)):

        indices = np.where(assignments == k)[0]

        cluster_points = X[indices]

        diff = cluster_points - centroids[k]

        distances = np.sum(
            diff * diff,
            axis=1
        )

        nearest_local_index = np.argmin(distances)

        original_index = indices[nearest_local_index]

        representatives.append({
            "cluster": k,
            "ID": ids[original_index],
            "distance_to_centroid":
                float(np.sqrt(distances[nearest_local_index]))
        })

    return representatives

In [54]:
representatives = representative_points(
    data_matrix_a,
    ids,
    result["centroids"],
    result["assignments"]
)

print(representatives)

[{'cluster': 0, 'ID': '8f20d2d155c74766a340c8966bf531f9', 'distance_to_centroid': 1.7118182182312012}, {'cluster': 1, 'ID': 'a064bc2a02ab456599a2186f40e07e0f', 'distance_to_centroid': 7.840341567993164}, {'cluster': 2, 'ID': '4d73a782967147ecbab8ad708d57c064', 'distance_to_centroid': 7.104903221130371}]
